# Game Phase Analysis — sfink37 (Bullet + Blitz, All Time)
**Project 1500 — Comparative Analysis**

All statistics computed fresh from game files on every run.
sfink37 plays primarily bullet and blitz — rapid games are excluded (only 11 exist).

In [ ]:
import chess
import chess.pgn
import json
import glob
import io
import re
from collections import Counter, defaultdict
from IPython.display import HTML, display
import matplotlib.pyplot as plt
import numpy as np

# ── Configuration ─────────────────────────────────────────────────────────────
USERNAME      = 'sfink37'
GAMES_DIR     = 'games_sf'
TIME_CLASSES  = {'bullet', 'blitz'}   # exclude rapid
EXCLUDE       = set()                  # no date exclusion — use all time
# ─────────────────────────────────────────────────────────────────────────────

OPENING_END    = 20   # half-moves (= move 10)
MIDDLEGAME_END = 50   # half-moves (= move 25)

## Step 1 — Load bullet + blitz games (all time)

In [ ]:
def load_games(username, games_dir, time_classes, exclude=set()):
    files = sorted(glob.glob(f'{games_dir}/*.json'))
    files = [f for f in files if not any(ex in f for ex in exclude)]
    games = []
    for f in files:
        with open(f, encoding='utf-8') as fh:
            month = json.load(fh)
        for g in month:
            if g.get('time_class') not in time_classes: continue
            white = g.get('white', {})
            black = g.get('black', {})
            if white.get('username','').lower() == username.lower():
                color, my, opp = 'white', white, black
            elif black.get('username','').lower() == username.lower():
                color, my, opp = 'black', black, white
            else:
                continue
            result = my.get('result', '')
            if   result == 'win': outcome = 'win'
            elif result in ('checkmated','timeout','resigned','lose','abandoned'): outcome = 'loss'
            elif result in ('agreed','stalemate','repetition','insufficient','timevsinsufficient','50move'): outcome = 'draw'
            else: continue
            eco_url = ''
            m = re.search(r'\[ECOUrl "([^"]+)"\]', g.get('pgn', ''))
            if m: eco_url = m.group(1)
            games.append({
                'outcome':    outcome,
                'color':      color,
                'end_reason': result,
                'time_class': g.get('time_class'),
                'pgn':        g.get('pgn', ''),
                'eco_url':    eco_url,
                'my_rating':  my.get('rating', 0),
                'opp_rating': opp.get('rating', 0),
                'end_time':   g.get('end_time', 0),
            })
    return sorted(games, key=lambda x: x['end_time'])

games  = load_games(USERNAME, GAMES_DIR, TIME_CLASSES, EXCLUDE)
losses = [g for g in games if g['outcome'] == 'loss']
total  = len(games)
wr     = 100 * sum(1 for g in games if g['outcome'] == 'win') / total

tc_counts = Counter(g['time_class'] for g in games)
print(f'Total games: {total}  (Win%: {wr:.1f}%)')
print(f'  Bullet: {tc_counts["bullet"]}  |  Blitz: {tc_counts["blitz"]}')
print(f'Total losses: {len(losses)}')

## Step 2 — Rating progression over time

In [ ]:
import datetime

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, tc, col in [(axes[0], 'bullet', '#c0392b'), (axes[1], 'blitz', '#2980b9')]:
    tc_games = [g for g in games if g['time_class'] == tc and g['my_rating'] > 0]
    if not tc_games: continue
    dates   = [datetime.datetime.fromtimestamp(g['end_time']) for g in tc_games]
    ratings = [g['my_rating'] for g in tc_games]
    ax.plot(dates, ratings, linewidth=0.7, alpha=0.8, color=col)
    ax.set_title(f'{USERNAME} — {tc.capitalize()} rating')
    ax.set_ylabel('Rating')
    if ratings:
        ax.set_ylim(min(ratings) - 50, max(ratings) + 50)
    ax.tick_params(axis='x', rotation=20)
plt.suptitle(f'{USERNAME}: Rating over time', fontsize=13)
plt.tight_layout()
plt.show()

## Step 3 — Assign game phase and count losses

In [ ]:
def count_halfmoves(pgn_str):
    try:
        game = chess.pgn.read_game(io.StringIO(pgn_str))
        return len(list(game.mainline_moves()))
    except Exception:
        return 0

def assign_phase(hm):
    if hm <= OPENING_END:    return 'Opening (<=move 10)'
    if hm <= MIDDLEGAME_END: return 'Middlegame (move 11-25)'
    return 'Endgame (move 26+)'

for g in games:
    g['halfmoves'] = count_halfmoves(g['pgn'])
    g['phase']     = assign_phase(g['halfmoves'])

losses = [g for g in games if g['outcome'] == 'loss']
phases = ['Opening (<=move 10)', 'Middlegame (move 11-25)', 'Endgame (move 26+)']
phase_counts = Counter(g['phase'] for g in losses)
total_losses = len(losses)

print(f'Losses by phase:')
for phase in phases:
    n   = phase_counts[phase]
    pct = 100 * n / total_losses
    print(f'  {phase:30s}: {n:4d}  ({pct:.1f}%)')

## Step 4 — Loss distribution and win rate by game length

In [ ]:
loss_lengths = [g['halfmoves'] for g in losses]
win_lengths  = [g['halfmoves'] for g in games if g['outcome'] == 'win']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(loss_lengths, bins=40, color='#c0392b', alpha=0.8, edgecolor='none')
axes[0].axvline(OPENING_END,    color='yellow',     linestyle='--', linewidth=1.5, label='Move 10')
axes[0].axvline(MIDDLEGAME_END, color='lightyellow', linestyle='--', linewidth=1.5, label='Move 25')
axes[0].set_xlabel('Half-moves played')
axes[0].set_ylabel('Losses')
axes[0].set_title(f'{USERNAME}: Loss distribution by game length')
axes[0].legend()

bins = range(0, max(loss_lengths + win_lengths) + 5, 3)
axes[1].hist(win_lengths,  bins=bins, color='#27ae60', alpha=0.6, label='Wins',   edgecolor='none')
axes[1].hist(loss_lengths, bins=bins, color='#c0392b', alpha=0.6, label='Losses', edgecolor='none')
axes[1].set_xlabel('Half-moves played')
axes[1].set_ylabel('Games')
axes[1].set_title('Wins vs Losses by game length')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Median loss length: {np.median(loss_lengths):.0f} half-moves ({np.median(loss_lengths)/2:.0f} full moves)')
print(f'Median win  length: {np.median(win_lengths):.0f} half-moves ({np.median(win_lengths)/2:.0f} full moves)')

In [ ]:
# Win rate by 5-move window
bucket_size  = 10
max_hm       = max(g['halfmoves'] for g in games)
bucket_stats = defaultdict(Counter)
for g in games:
    b = (g['halfmoves'] // bucket_size) * bucket_size
    bucket_stats[b][g['outcome']] += 1

xs, wrs, totals = [], [], []
for b in sorted(bucket_stats):
    c = bucket_stats[b]
    t = sum(c.values())
    if t < 5: continue
    xs.append(b // 2)
    wrs.append(100 * c['win'] / t)
    totals.append(t)

fig, ax = plt.subplots(figsize=(13, 4))
colors = ['#27ae60' if wr >= 50 else '#c0392b' for wr in wrs]
bars   = ax.bar(xs, wrs, width=4, color=colors, edgecolor='none', alpha=0.85)
ax.axhline(50, color='gray', linestyle='--', linewidth=1)
ax.axvline(10, color='yellow',     linestyle=':', linewidth=1.5, label='Move 10')
ax.axvline(25, color='lightyellow', linestyle=':', linewidth=1.5, label='Move 25')
ax.set_xlabel('Full move number (5-move windows)')
ax.set_ylabel('Win %')
ax.set_title(f'{USERNAME}: Win rate by game length')
ax.legend()
for bar, wr, n in zip(bars, wrs, totals):
    if n >= 15:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f'{wr:.0f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## Step 5 — How do losses end, by phase

In [ ]:
def classify_end(reason):
    if reason == 'checkmated':                 return 'Checkmate'
    if reason in ('resigned', 'lose'):         return 'Resignation'
    if reason in ('timeout', 'abandoned'):     return 'Timeout'
    return 'Other'

end_types  = ['Checkmate', 'Resignation', 'Timeout', 'Other']
colors_end = ['#c0392b', '#e67e22', '#8e44ad', '#888']
phase_end  = {ph: Counter() for ph in phases}
for g in losses:
    phase_end[g['phase']][classify_end(g['end_reason'])] += 1

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, phase in zip(axes, phases):
    c     = phase_end[phase]
    total = sum(c.values())
    ax.bar(end_types, [c[et] for et in end_types], color=colors_end, edgecolor='none')
    ax.set_title(f'{phase}\n({total} losses)')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=20)
plt.suptitle(f'{USERNAME}: How do losses end — by phase?', fontsize=13)
plt.tight_layout()
plt.show()

## Step 6 — Bullet vs Blitz comparison

In [ ]:
for tc in ['bullet', 'blitz']:
    tc_games  = [g for g in games  if g['time_class'] == tc]
    tc_losses = [g for g in losses if g['time_class'] == tc]
    if not tc_games: continue
    tc_wr = 100 * sum(1 for g in tc_games if g['outcome'] == 'win') / len(tc_games)
    print(f'{tc.capitalize()} ({len(tc_games)} games, {tc_wr:.1f}% win rate):')
    for phase in phases:
        n   = sum(1 for g in tc_losses if g['phase'] == phase)
        pct = 100 * n / len(tc_losses) if tc_losses else 0
        print(f'  {phase:30s}: {n:3d} losses  ({pct:.0f}%)')
    print()

## Step 7 — Summary

In [ ]:
opening_pct    = 100 * phase_counts['Opening (<=move 10)']    / total_losses
middlegame_pct = 100 * phase_counts['Middlegame (move 11-25)'] / total_losses
endgame_pct    = 100 * phase_counts['Endgame (move 26+)']      / total_losses
dominant_phase = max(phase_counts, key=phase_counts.get)

print(f'{USERNAME.upper()} — GAME PHASE SUMMARY')
print('=' * 50)
print(f'Total games:    {total}  ({wr:.1f}% win rate)')
print(f'Total losses:   {total_losses}')
print()
print(f'  Opening losses:    {phase_counts["Opening (<=move 10)"]:4d}  ({opening_pct:.1f}%)')
print(f'  Middlegame losses: {phase_counts["Middlegame (move 11-25)"]:4d}  ({middlegame_pct:.1f}%)')
print(f'  Endgame losses:    {phase_counts["Endgame (move 26+)"]:4d}  ({endgame_pct:.1f}%)')
print(f'\nDominant loss phase: {dominant_phase}')
print()
if opening_pct > 35:
    print('>> Opening losses are significant — opening study warranted.')
if middlegame_pct > 40:
    print('>> Middlegame is the main problem — tactical/positional training needed.')
if endgame_pct > 30:
    print('>> Endgame losses are significant — endgame technique study warranted.')
if phase_counts.get('Timeout/Abandoned losses', 0) > total_losses * 0.15:
    print('>> Time management is a significant factor.')